# AI Integrations for Developers — Exam

## Instructions

- This notebook is a **template** where you must put your code.  
- You should **fill in all empty variables** and complete the code so that when I download your notebook and click **Run all**, all cells execute correctly and provide the answers.  
- ⚠️ **Do NOT hardcode your API key**. Use Colab environment variables (`%env OPENAI_API_KEY=your_key_here`) and access them in your code.  
- You may **create more cells** if needed. It is recommended that your code is well-structured and split logically into separate cells.  
- The function **`ask_ai(query)`** must be implemented by you. All queries will call this function to check your solution.  
- ✅ **Test cases will be created by me (the instructor).** You are **not allowed to modify, remove, or add to the test cases cell**. Your code must work correctly with the provided test cases.  
- You are **ONLY ALLOWED** to use only the following:  
  - **Models:** OpenAI or Anthropic  
  - **Technologies:** LangChain or vanilla Python code  
  - **Vector Store:** Chroma DB

🚨 **Any student who does not follow the template, does not stick to the required format, or whose code does not execute properly will be disqualified.**


### Important

Fill in **all the variables** in the cell.  
❌ **Do NOT put your API key directly in the code.**  
✅ The cell must be set up to take the API key from the Colab environment variables.


In [104]:
# ================================
# 🔧 RAG Configuration Variables
# ================================

# ⚠️ Do NOT put your API key here directly.
# Make sure you set your API key in Colab like this:
# %env OPENAI_API_KEY=your_key_here

import os
from google.colab import userdata

# API Key (taken from Colab environment variables)
API_KEY = userdata.get("OPENAI_API_KEY")

os.environ["OPENAI_API_KEY"] = API_KEY

# Prompt & Model Settings
SYSTEM_MESSAGE_CONVERSATION_ANALYST = """
<context>
</context>

<role>
</role>

<who_am_I>
</who_am_I>
"""

SYSTEM_MESSAGE_RESPONSE_OPTIMIZER = """
<task>
1. Analyze the TEXT TO BE OPTIMIZED.
2. Do not change its intended meaning.
3. If the text is cut off mid-sentence, mid-thought, or mid-paragraph, shorten or trim it so it ends logically and completely.
</task>
<next>
Output only the optimized text.
</next>
"""

HUMAN_MESSAGE_RESPONSE_OPTIMIZER = """
TEXT TO BE OPTIMIZED:\n{text}
"""

SYSTEM_MESSAGE_RESPONDER = """
You are a helpful assistant.
<critical_rule>
End with a complete sentence without cutting off mid-thought, mid-sentence or mid-paragraph.
</critical_rule>
"""

HUMAN_MESSAGE = """
CONVERSATION MEMORY:\n{conversation_memory}\n\n
Based on the following CONTEXT:\n{context}\n\n
Answer the following QUESTION:\n{query}
"""

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# Chunking Parameters
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
TOP_N_RESULTS = 4

# Generation Parameters
OUTPUT_LENGTH = 400
TEMPERATURE = 0.1
TOP_P = 0.1
FREQUENCY_PENALTY = 1.0
PRESENCE_PENALTY = 1.0

### Code Organization

Create more cells if needed and put your code in them.  
It is **recommended** that your code is well-structured, split logically, and kept in separate cells for clarity.


## **Section 1: Setup**

### 1. Install Dependencies

In [105]:
!pip install -U langchain langchain-openai langchain-community chromadb pypdf python-dotenv

### 2. Upload PDF to Colab

In [106]:
from google.colab import files

uploaded_file = files.upload()

Saving Exam_Preparation_PDF.pdf to Exam_Preparation_PDF (9).pdf


### 3. Load the PDF

In [107]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = list(uploaded_file.keys())[0]
loader = PyPDFLoader(pdf_path)

pages = loader.load()

### 4. Chunk the PDF

In [108]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=CHUNK_SIZE,
      chunk_overlap=CHUNK_OVERLAP,
      separators=[ "\n\n", "\n", ". ", "!", "?", ", ", "; ", ": ", "-"],
  )

chunks = text_splitter.split_documents(pages)

### 5. Initialize embedding model

In [109]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
     model=EMBEDDING_MODEL
  )

### 6. Initialize vectorstore

In [110]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=None,
  )

### 7. Initialize Conversation Buffer Memory

In [111]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="conversation_memory",
    return_messages=True,
    output_key="response"
)

### 8. Initialize the Chat Model

In [112]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=MODEL,
    max_tokens=OUTPUT_LENGTH,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    frequency_penalty=FREQUENCY_PENALTY,
    presence_penalty=PRESENCE_PENALTY,
    streaming=True,
)

## **Section 2: Handler Functions**

### 1. Retrieve relevant content from database

In [113]:
def retrieve_relevant_context(
    vectorstore,
    query,
    k=4
):
    results = vectorstore.similarity_search(
        query,
        k=k
    )
    context = '\n'.join(
        result.page_content for result in results
    )

    return context.strip()

### 2. Generate AI response

In [114]:
from langchain.prompts import ChatPromptTemplate
from langchain.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

def generate_ai_response(
    llm,
    system_message,
    human_message,
    **kwargs
):
    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(
            system_message
        ),
        HumanMessagePromptTemplate.from_template(
            human_message
        ),
    ])

    messages = prompt.format_messages(**kwargs)

    return llm.invoke(messages).content

## Test Cases (Final Cell)

The final cell must contain your **test cases**.  
When executed, the AI should provide correct answers to the given questions **based on the PDF file**.


### AI Query Function

In this cell, you must implement the function **ask_ai(query)**.  
This function will be the final execution point of your pipeline (RAG / LLM).  


In [115]:
# ================================
# ❓ AI Query Function
# ================================

def ask_ai(query: str):
    """
    This function should execute your final RAG / LLM pipeline.
    Input:
        query (str): The question you want to ask the AI.
    Output:
        str: The AI's answer based on the PDF file.
    """
    # TODO: Implement your final execution logic here
    # Example steps:
    # 1. Retrieve relevant chunks
    # 2. Generate embeddings
    # 3. Call the model with your prompt + retrieved context
    # 4. Return the model's answer

    context = retrieve_relevant_context(
      vectorstore,
      query,
      TOP_N_RESULTS,
    )

    ai_response = generate_ai_response(
      llm,
      SYSTEM_MESSAGE_RESPONDER,
      HUMAN_MESSAGE,
      conversation_memory=memory.load_memory_variables({})['conversation_memory'],
      context=context,
      query=query,
    )

    memory.save_context({"input": query}, {"response": ai_response})


    return ai_response

### Test Queries

Use this cell to test your function with different queries.  
The answers must be generated correctly based on the PDF file.  


In [116]:
# ================================
# 🔍 Example Queries for Testing
# ================================

queries = [
    "How many words should effective prompts average?",
    "List the four main areas for effective prompts.",
    "What does 'persona' mean in prompt writing?",
    "Name three business roles covered in this guide.",
    "What is Gemini Advanced?",
    "My name is Bea",
    "What is my name?"
]

# Call the AI with each query
for q in queries:
    print(f"Q: {q}")
    print(f"A: {ask_ai(q)}\n")


Q: How many words should effective prompts average?
A: Effective prompts should average around 21 words.

Q: List the four main areas for effective prompts.
A: The four main areas for effective prompts are: 

1. Persona
2. Task
3. Context
4. Format

Q: What does 'persona' mean in prompt writing?
A: In prompt writing, 'persona' refers to the character or identity that the AI is expected to adopt when responding. This includes defining traits, tone, and style that align with the intended audience or purpose of the interaction. Establishing a clear persona helps guide how responses are framed and ensures they resonate appropriately with users.

Q: Name three business roles covered in this guide.
A: Three business roles covered in this guide are executives, human resources, and marketing.

Q: What is Gemini Advanced?
A: Gemini Advanced is a sophisticated AI model developed to enhance various applications, providing advanced capabilities in natural language processing and understanding. It 